In [1]:
'''Neural Network (CNN 구조 실습)
Decision Boundary
Gradient Descent 실습
'''
import torch
import torch.nn as nn
import torch.nn.functional as F


class Net(nn.Module):
    def __init__(self):
        super().__init__()

        # conv = 이미지에서 중요한 모양(특징) 찾는 필터(쿠키틀 느낌)
        self.conv1 = nn.Conv2d(1, 6, 5)   # 흑백(1채널) → 특징지도 6개
        self.conv2 = nn.Conv2d(6, 16, 5)  # 특징지도 6개 → 16개

        # conv+pool 끝나면 최종 feature map 크기가 (16, 5, 5)가 됨
        # 그래서 flatten하면 16*5*5 = 400
        self.fc1 = nn.Linear(16 * 5 * 5, 120)
        self.fc2 = nn.Linear(120, 84)
        self.fc3 = nn.Linear(84, 10)  # 10개 클래스 점수(logits)

    def forward(self, x):
        # shape: (N, C, H, W)
        # N: 이미지 몇 장(배치), C: 채널, H/W: 높이/너비

        # 1) conv로 특징 찾기 + relu로 “복잡한 모양도 구분 가능하게” 만들기
        x = F.relu(self.conv1(x))
        # 2) pooling으로 크기 줄이기(요약)
        x = F.max_pool2d(x, 2)

        x = F.relu(self.conv2(x))
        x = F.max_pool2d(x, 2)

        # 3) (N,16,5,5) → (N,400)으로 펴기
        x = torch.flatten(x, 1)

        # 4) fc로 “무슨 클래스인지” 점수 만들기
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = self.fc3(x)  # logits(softmax 전 점수)

        return x


# ===== 모델 동작 확인 =====
net = Net()
x_test = torch.randn(1, 1, 32, 32)
y_test = net(x_test)
print("출력 shape:", y_test.shape)  # (1,10)


# ===== 학습(Gradient Descent) + Decision Boundary 연결 =====
# Decision Boundary:
# - fc3가 만든 “클래스 점수(logits)” 비교로 클래스가 결정됨 (argmax)
# - step으로 weight가 바뀌면 점수 계산 방식이 바뀌고, 그럼 경계도 움직임

criterion = nn.CrossEntropyLoss()              # 정답과 예측 차이 계산(softmax 포함)
optimizer = torch.optim.SGD(net.parameters(), lr=0.01)  # 경사하강법

x = torch.randn(1, 1, 32, 32)
target = torch.tensor([3])

for step in range(3):
    output = net(x)                 # (1) forward
    loss = criterion(output, target) # (2) loss(오차)

    optimizer.zero_grad()           # (3) 이전 기울기 지우기
    loss.backward()                 # (4) 기울기 계산
    w_before = net.fc3.weight.data[0, 0].item()

    optimizer.step()                # (5) weight 업데이트(=경사하강법)
    w_after = net.fc3.weight.data[0, 0].item()

    print(f"step {step} loss:", loss.item())
    print("fc3.weight[0,0] before:", w_before, "after:", w_after)

# (확인용) 확률/예측 클래스 보기
probs = torch.softmax(output, dim=1)
print("예측 클래스:", probs.argmax(dim=1).item())
print("확률:", probs)


출력 shape: torch.Size([1, 10])
step 0 loss: 2.2505147457122803
fc3.weight[0,0] before: 0.06576075404882431 after: 0.06576075404882431
step 1 loss: 2.210620164871216
fc3.weight[0,0] before: 0.06576075404882431 after: 0.06576075404882431
step 2 loss: 2.168727159500122
fc3.weight[0,0] before: 0.06576075404882431 after: 0.06576075404882431
예측 클래스: 3
확률: tensor([[0.0986, 0.0918, 0.1006, 0.1143, 0.1052, 0.0929, 0.1031, 0.0965, 0.1029,
         0.0940]], grad_fn=<SoftmaxBackward0>)
